# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

We start from the raw starter CSV and build a clean feature vector by:
- Filling missing numerics with 0 and categoricals with `'unknown'`
- Log-transforming heavy-tailed traffic counts
- Adding binary indicator flags for important sparse signals
- Fixing the `avg_position = 0` gotcha (0 means no data, not rank zero)
- Filtering to pages with `impressions_90d > 0` and `content_age_days >= 90`

**Crucially**, `trend_direction`, `trend_pct`, `impressions_last_30d`, `clicks_last_30d`, and `sessions_last_30d` are **not** in this feature vector — they define or directly compute the label.

In [1]:
import os
import numpy as np
import pandas as pd

# ── Robust path resolution ─────────────────────────────────────────────────
paths = [
    'data/raw/content_refresh_anonymized.csv',
    '../data/raw/content_refresh_anonymized.csv',
    '../../data/raw/content_refresh_anonymized.csv',
]
df_raw = None
for p in paths:
    if os.path.exists(p):
        df_raw = pd.read_csv(p)
        print(f'Loaded from: {p}')
        break
if df_raw is None:
    raise FileNotFoundError('Could not find content_refresh_anonymized.csv')

# ── Step 1: Filter to valid rows ───────────────────────────────────────────
df = df_raw[(df_raw['impressions_90d'] > 0) & (df_raw['content_age_days'] >= 90)].copy()
df = df.drop_duplicates(subset=['content_id']).reset_index(drop=True)
print(f'Rows after filter: {len(df):,}')

# ── Step 2: Define the label FIRST, then remove label-source columns ───────
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

LABEL_SOURCE_COLS = ['trend_direction', 'trend_pct',
                     'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d']

# ── Step 3: Fix avg_position = 0 (means no position data, not rank 0) ─────
df['avg_position'] = df['avg_position'].replace(0, np.nan)
df['has_position'] = df['avg_position'].notna().astype(int)
df['avg_position'] = df['avg_position'].fillna(100)  # fill with a high-rank sentinel

# ── Step 4: Fill missing numerics ─────────────────────────────────────────
numeric_fill_zero = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d',
    'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
    'days_with_impressions', 'days_with_sessions',
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d',
    'content_age_days', 'age_tier_order', 'days_since_last_update',
    'ctr', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
]
for col in numeric_fill_zero:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)

# ── Step 5: Fill missing categoricals ─────────────────────────────────────
categorical_cols = [
    'competition_level', 'content_type', 'main_intent',
    'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier',
    'impression_tier', 'position_tier',
]
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].fillna('unknown').astype(str).replace({'': 'unknown', 'nan': 'unknown'})

# ── Step 6: Log-transform heavy-tailed traffic columns ────────────────────
df['log_impressions_90d'] = np.log1p(df['impressions_90d'])
df['log_clicks_90d']      = np.log1p(df['clicks_90d'])
df['log_sessions_90d']    = np.log1p(df['sessions_90d'])
df['log_ai_sessions_90d'] = np.log1p(df['ai_sessions_90d'])

# ── Step 7: Indicator flags for sparse signals ────────────────────────────
df['has_clicks']      = (df['clicks_90d'] > 0).astype(int)
df['has_ai_sessions'] = (df['ai_sessions_90d'] > 0).astype(int)
df['has_word_count']  = (df['word_count'] > 0).astype(int)  # avoids feedly leakage
df['measurable_opportunity'] = (
    (df['impressions_90d'] >= 100) & (df['sessions_90d'] > 0)
).astype(int)

# ── Final model features ──────────────────────────────────────────────────
MODEL_NUMERIC = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d',
    'days_with_impressions', 'days_with_sessions',
    'content_age_days', 'days_since_last_update',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
    'has_clicks', 'has_ai_sessions', 'has_word_count', 'has_position',
    'measurable_opportunity',
]
MODEL_CATEGORICAL = [
    'competition_level', 'content_type', 'main_intent',
    'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier',
]

ALL_FEATURES = MODEL_NUMERIC + MODEL_CATEGORICAL
feature_df = df[['content_id', 'client_id'] + ALL_FEATURES + ['is_declining_label']].copy()

print(f'Feature matrix shape: {feature_df.shape}')
print(f'Label rate (declining): {feature_df["is_declining_label"].mean():.2%}')
print(f'\nNumeric features ({len(MODEL_NUMERIC)}): {MODEL_NUMERIC}')
print(f'\nCategorical features ({len(MODEL_CATEGORICAL)}): {MODEL_CATEGORICAL}')


Loaded from: ../../data/raw/content_refresh_anonymized.csv
Rows after filter: 30,000
Feature matrix shape: (30000, 34)
Label rate (declining): 54.21%

Numeric features (23): ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'has_clicks', 'has_ai_sessions', 'has_word_count', 'has_position', 'measurable_opportunity']

Categorical features (8): ['competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']


## 2. Feature notes (meaning, missing, categorical, available-when?)

| Feature | Meaning | Missing Handling | Available Before Prediction? |
|---|---|---|---|
| `search_volume` | Keyword search volume estimate | Fill 0 (no keyword data) | ✅ Yes — keyword metadata |
| `competition` | Keyword competition 0–1 | Fill 0 | ✅ Yes |
| `cpc` | Cost-per-click estimate | Fill 0 | ✅ Yes |
| `word_count` | Article word count | Fill 0 + `has_word_count` flag | ✅ Yes — content property |
| `char_count` | Article character count | Fill 0 | ✅ Yes |
| `log_impressions_90d` | log1p(GSC impressions over 90 days) | None (all rows have ≥1) | ✅ Yes — historical signal |
| `log_clicks_90d` | log1p(GSC clicks over 90 days) | None | ✅ Yes |
| `log_sessions_90d` | log1p(GA4 sessions over 90 days) | None | ✅ Yes |
| `log_ai_sessions_90d` | log1p(AI-referred sessions) | None | ✅ Yes |
| `days_with_impressions` | Days in window with ≥1 impression | None | ✅ Yes |
| `days_with_sessions` | Days in window with ≥1 session | None | ✅ Yes |
| `content_age_days` | Days since page was created | None (all ≥90) | ✅ Yes |
| `days_since_last_update` | Days since last content edit | None | ✅ Yes |
| `ctr` | clicks/impressions × 100 (0.76 = 0.76%) | Fill 0 | ✅ Yes — historical derived |
| `avg_position` | Mean GSC rank (0→100 sentinel, `has_position` flag) | Replace 0 with 100 | ✅ Yes |
| `engagement_rate` | engaged_sessions/sessions × 100 | Fill 0 | ✅ Yes |
| `scroll_rate` | scroll_events/pageviews × 100 (can exceed 100) | Fill 0 | ✅ Yes |
| `ai_traffic_pct` | ai_sessions/sessions × 100 (can exceed 100) | Fill 0 | ✅ Yes |
| `has_clicks` | Binary: page had any clicks | None | ✅ Yes |
| `has_ai_sessions` | Binary: page had any AI-referred sessions | None | ✅ Yes |
| `has_word_count` | Binary: word_count was measured | None | ✅ Yes — avoids feedly type leaking |
| `has_position` | Binary: avg_position data exists | None | ✅ Yes |
| `measurable_opportunity` | impressions≥100 AND sessions>0 | None | ✅ Yes |
| `competition_level` | LOW/MEDIUM/HIGH keyword tier | Fill 'unknown' | ✅ Yes |
| `content_type` | keyword article / feedly article / comparison article | None | ✅ Yes |
| `main_intent` | informational / transactional / commercial / navigational | Fill 'unknown' | ✅ Yes |
| `age_tier` | Bucketed content age | None | ✅ Yes |
| `freshness_tier` | Bucketed days since update | None | ✅ Yes |
| `word_count_tier` | Bucketed word count | Fill 'unknown' | ✅ Yes |
| `impression_tier` | no_data / none / low / moderate / good / excellent | None | ✅ Yes |
| `position_tier` | top_3 / page_1 / striking / page_3_5 / deep | None | ✅ Yes |

**Key note on systematic missingness**: `word_count`, `word_count_tier`, `char_count`, `char_count_tier`, and keyword columns are **structurally missing for `feedly article` and `comparison article`** content types. A plain `fillna(0)` would silently encode `content_type` into those columns. We mitigate this with the `has_word_count` flag so the model can distinguish "0 words" from "unmeasured".

In [2]:
# Confirm zero nulls in the final feature matrix
null_counts = feature_df[ALL_FEATURES].isnull().sum()
print('Null counts in feature matrix (must all be 0):')
print(null_counts[null_counts > 0] if null_counts.any() else '  [OK] No nulls - all clean.')

# Show missingness pattern by content_type for key columns
print('\nMissingness in raw data by content_type (word_count and search_volume):')
print(
    df_raw.groupby('content_type')[['word_count', 'search_volume']]
    .apply(lambda g: g.isnull().mean() * 100)
    .round(1)
)


Null counts in feature matrix (must all be 0):
  [OK] No nulls - all clean.

Missingness in raw data by content_type (word_count and search_volume):
                    word_count  search_volume
content_type                                 
comparison article         0.0            0.0
feedly article             0.0          100.0
keyword article           28.3            1.4


## 3. The leakage hunt

We systematically attack our own feature vector across the three leakage types:

### Type 1 — Label-derived features
The label `is_declining_label = (trend_direction == 'down')` is derived from:
- `trend_direction` (direct source of the label) → **EXCLUDED**
- `trend_pct` (the numeric signal trend_direction is bucketed from) → **EXCLUDED**

**Test**: Train a trivial model WITH `trend_direction` as a feature → expect near-perfect precision. Without it → score drops to honest level.

### Type 2 — Future/overlapping windows
The trend label compares `impressions_last_30d` vs `impressions_prev_30d`. Therefore:
- `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d` → **EXCLUDED** (they are literally in the label formula)
- `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d` → intentionally **also excluded** from the final feature set (they feed into the trend computation on the same 90-day window — using them gives the model the raw ingredients to reconstruct the label)

Timeline diagram:
```
|------- prev_30d (days 31-60) ------|------ last_30d (days 1-30) ------ | TODAY
                                       ↑ label is computed here            ↑
features must NOT use anything in last_30d window
```

### Type 3 — Decision-derived / product flags
- `provider_used`, `model_used` → metadata about which LLM generated the content, not search/engagement signal. **EXCLUDED**.
- `impression_tier`, `position_tier` — these are deterministic buckets derived entirely from features already in the vector (`impressions_90d`, `avg_position`). They add no new information but could confuse importance. We keep `impression_tier` and `position_tier` as they provide ordinal context, but acknowledge they are derived.

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OrdinalEncoder
import numpy as np

def build_X(frame, features_num, features_cat, fit_encoder=None):
    """Encode features into a numpy array for sklearn."""
    X_num = frame[features_num].values.astype(float)
    if features_cat:
        X_cat_raw = frame[features_cat].astype(str).values
        if fit_encoder is None:
            enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
            X_cat = enc.fit_transform(X_cat_raw)
        else:
            enc = fit_encoder
            X_cat = enc.transform(X_cat_raw)
        return np.hstack([X_num, X_cat]), enc
    return X_num, None

def precision_at_k(y_true, scores, k=50):
    idx = np.argsort(scores)[::-1][:k]
    return float(y_true[idx].mean()) if len(idx) else 0.0

y = feature_df['is_declining_label'].values
groups = feature_df['client_id'].values
base_rate = y.mean()

# ── Grouped split (honest: train on some clients, test on others) ──────────
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(feature_df, y, groups=groups))

train_df = feature_df.iloc[train_idx]
test_df  = feature_df.iloc[test_idx]
y_train  = y[train_idx]
y_test   = y[test_idx]

print(f'Base rate (label rate): {base_rate:.2%}')
print(f'Train: {len(train_df):,} rows | Test: {len(test_df):,} rows')
print(f'Test clients: {test_df["client_id"].nunique()} (never seen in training)')

# ── HONEST model (no leaky features) ──────────────────────────────────────
X_train, enc = build_X(train_df, MODEL_NUMERIC, MODEL_CATEGORICAL)
X_test,  _   = build_X(test_df,  MODEL_NUMERIC, MODEL_CATEGORICAL, fit_encoder=enc)

clf_honest = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf_honest.fit(X_train, y_train)
scores_honest = clf_honest.predict_proba(X_test)[:, 1]
p50_honest = precision_at_k(y_test, scores_honest, k=50)
print(f'[OK] Honest model Precision@50: {p50_honest:.3f}  (base rate = {base_rate:.2%})')

# ── LEAKY model (with trend_direction as feature — the confession test) ────
# encode trend_direction as a numeric: 'down'=0, others=1
leaky_col = (df.loc[feature_df.index, 'trend_direction'] == 'down').astype(int)
# Note: we use df not feature_df here because we explicitly excluded it
# In practice the leaky feature perfectly predicts the label by definition
X_train_leaky = np.hstack([X_train, leaky_col.iloc[train_idx].values.reshape(-1, 1)])
X_test_leaky  = np.hstack([X_test,  leaky_col.iloc[test_idx].values.reshape(-1, 1)])

clf_leaky = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf_leaky.fit(X_train_leaky, y_train)
scores_leaky = clf_leaky.predict_proba(X_test_leaky)[:, 1]
p50_leaky = precision_at_k(y_test, scores_leaky, k=50)
print(f'[LEAKY] Leaky model Precision@50: {p50_leaky:.3f}  (with trend_direction as feature)')
print(f'   Leakage uplift: +{(p50_leaky - p50_honest):.3f} — confirms trend_direction is label-derived')


Base rate (label rate): 54.21%
Train: 19,166 rows | Test: 10,834 rows
Test clients: 10 (never seen in training)


[OK] Honest model Precision@50: 0.680  (base rate = 54.21%)


[LEAKY] Leaky model Precision@50: 1.000  (with trend_direction as feature)
   Leakage uplift: +0.320 — confirms trend_direction is label-derived


## 4. What I excluded and why

| Excluded Column | Reason |
|---|---|
| `trend_direction` | **Label source**: `is_declining_label = (trend_direction == 'down')`. Using it is 100% label leakage. |
| `trend_pct` | **Label-derived**: `trend_pct` is the numeric signal from which `trend_direction` is thresholded. Including it lets the model reconstruct the label exactly. |
| `impressions_last_30d` | **Window overlap**: directly used in the label formula `(last_30d − prev_30d) / prev_30d`. |
| `clicks_last_30d` | **Window overlap**: same label window. Including it lets the model see the numerator of the future trend. |
| `sessions_last_30d` | **Window overlap**: same label window. |
| `impressions_prev_30d` | **Excluded by choice**: also feeds the trend formula denominator; keeping it gives the model the raw ingredients to reconstruct the label signal. |
| `clicks_prev_30d` | **Excluded by choice**: same reasoning as `impressions_prev_30d`. |
| `sessions_prev_30d` | **Excluded by choice**: same reasoning. |
| `provider_used` | **Metadata / product flag**: which LLM generated the article has no causal link to search performance; including it would teach the model about which AI tool was used, not the content itself. |
| `model_used` | **Metadata / product flag**: same as `provider_used`. High cardinality, mostly missing, no performance signal. |
| `content_id` | **Context only**: pseudonymous identifier used for deduplication and grouping, never a feature. |
| `client_id` | **Context only**: used for grouped train/test splits. Encoding it as a feature would let the model memorize client identity rather than learn generalizable signals. |

In [4]:
EXCLUDED = {
    'trend_direction':       'Label source — is_declining_label is derived from this column.',
    'trend_pct':             'Label-derived — the numeric behind trend_direction.',
    'impressions_last_30d':  'Window overlap — in the label formula (last-30d impressions).',
    'clicks_last_30d':       'Window overlap — in the label window.',
    'sessions_last_30d':     'Window overlap — in the label window.',
    'impressions_prev_30d':  'Feeds trend denominator — gives model raw label ingredients.',
    'clicks_prev_30d':       'Feeds trend denominator — same reason.',
    'sessions_prev_30d':     'Feeds trend denominator — same reason.',
    'provider_used':         'Metadata/product flag — no causal search signal.',
    'model_used':            'Metadata/product flag — same as provider_used.',
    'content_id':            'Context only — deduplication and grouping.',
    'client_id':             'Context only — used for grouped splits, not a feature.',
}

print('Excluded columns and rationale:')
for col, reason in EXCLUDED.items():
    present = '(in raw data)' if col in df_raw.columns else '(not in raw data)'
    print(f'  {col:<28} {present:<16} → {reason}')

# Verify none of the excluded columns are in the model feature set
leaked = [c for c in EXCLUDED if c in ALL_FEATURES]
if leaked:
    raise AssertionError(f'LEAKAGE DETECTED — excluded columns found in features: {leaked}')
else:
    print('\n[OK] Confirmed: no excluded columns appear in the model feature set.')


Excluded columns and rationale:
  trend_direction              (in raw data)    → Label source — is_declining_label is derived from this column.
  trend_pct                    (in raw data)    → Label-derived — the numeric behind trend_direction.
  impressions_last_30d         (in raw data)    → Window overlap — in the label formula (last-30d impressions).
  clicks_last_30d              (in raw data)    → Window overlap — in the label window.
  sessions_last_30d            (in raw data)    → Window overlap — in the label window.
  impressions_prev_30d         (in raw data)    → Feeds trend denominator — gives model raw label ingredients.
  clicks_prev_30d              (in raw data)    → Feeds trend denominator — same reason.
  sessions_prev_30d            (in raw data)    → Feeds trend denominator — same reason.
  provider_used                (in raw data)    → Metadata/product flag — no causal search signal.
  model_used                   (in raw data)    → Metadata/product flag — sam

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.